# Power Demand Forecasting
## Forecasting Daily Electricity Demand (AEP Region)

**Notebook #16 of 50 — Time Series Forecasting Portfolio**

| | |
|---|---|
| **Dataset** | Hourly Energy Consumption (`robikscube/hourly-energy-consumption`) |
| **File** | `AEP_hourly.csv` |
| **Columns** | `Datetime`, `AEP_MW` |
| **Raw frequency** | Hourly (resampled to Daily mean for 14-day horizon) |
| **Target** | Daily mean electricity demand (MW) |
| **TS Library** | Darts (ExponentialSmoothing + AutoARIMA) |

## Learning Objectives
1. Resample hourly electricity data to daily mean demand without information leakage
2. Identify multi-scale seasonality: daily (within-day), weekly (work vs weekend)
3. Apply Darts for unified classical + neural time series experiments
4. Understand the operational importance of power demand forecasting (grid balancing)
5. Discuss the cost asymmetry: under-generation causes blackouts; over-generation wastes fuel

## Why This Project Matters
Electricity is unique among commodities — it **cannot be stored** at grid scale (economically).
Generation must match consumption in real time, every second of every day.
A 14-day demand forecast is critical for:
- **Power plant dispatch**: deciding which coal/gas/nuclear plants to bring online (12–24 hour startup)
- **Fuel procurement**: ordering natural gas days ahead at favourable spot prices
- **Renewable integration**: sizing the dispatchable backup for intermittent solar/wind
- **Grid reliability**: maintaining 15–20% reserve margin to prevent rolling blackouts

Forecast errors of even 3–5% can cost millions in fuel waste or, worse, trigger load shedding.

## Problem Statement
Electricity demand cannot be stored efficiently — generation must match consumption
in real time. Forecasting daily demand is essential for:
- **Power plant dispatch scheduling** (which plants to run tomorrow?)
- **Capacity reserve planning** (maintain 15-20% headroom)
- **Renewable integration** (complement solar/wind with dispatchable backup)
- **Spot price forecasting** (price tracks marginal generation cost)

> Goal: Forecast daily mean electricity demand 14 days ahead for the AEP service region.

## Dataset Overview
**Source:** Kaggle — `robikscube/hourly-energy-consumption`

**License:** CC0 Public Domain

### Key details
- **AEP** = American Electric Power, a large Midwestern US utility
- **AEP_MW** = Hourly energy consumption in megawatts
- **Datetime** = UTC-based hourly timestamps
- Dataset spans **2004–2018** (~13 years, 121k+ rows)
- We resample to **daily mean MW** for our forecasting horizon

### Seasonal patterns to expect
| Pattern | Period | Driver |
|---------|--------|--------|
| Weekly | 7 days | Weekdays > Weekends (industrial load) |
| Annual | 365 days | Summer AC + Winter heating peaks |
| Long-term trend | ~Decade | Economic growth vs efficiency gains |

## Environment Setup

In [ ]:
import subprocess, sys
for imp, pkg in {"kagglehub":"kagglehub","darts":"darts","statsforecast":"statsforecast",
                  "mlforecast":"mlforecast","lightgbm":"lightgbm",
                  "lazypredict":"lazypredict","flaml":"flaml[automl]"}.items():
    try: __import__(imp)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Packages ready.")

## Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from sklearn.metrics import mean_absolute_error, mean_squared_error
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive
from mlforecast import MLForecast
from lightgbm import LGBMRegressor
from window_ops.rolling import rolling_mean

from darts import TimeSeries
from darts.models import ExponentialSmoothing, AutoARIMA as DartsAutoARIMA, NaiveSeasonal
from darts.metrics import mae as darts_mae, rmse as darts_rmse

pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.figsize": (14, 5), "axes.grid": True})
sns.set_theme(style="whitegrid")

def metrics(actual, pred, label):
    a = np.asarray(actual).ravel(); p = np.asarray(pred).ravel()
    n = min(len(a), len(p)); a, p = a[:n], p[:n]
    mad = mean_absolute_error(a, p)
    rmse = np.sqrt(mean_squared_error(a, p))
    mask = a != 0
    mape = np.mean(np.abs((a[mask]-p[mask])/a[mask]))*100 if mask.sum()>0 else float("nan")
    print(f"{label:<42s}  MAE={mad:>10,.2f}  RMSE={rmse:>10,.2f}  MAPE={mape:>7.2f}%")
    return {"Model":label,"MAE":round(mad,2),"RMSE":round(rmse,2),"MAPE":round(mape,2)}
print("Imports OK.")

## Configuration

In [ ]:
KAGGLE_SLUG  = "robikscube/hourly-energy-consumption"
TARGET_FILE  = "AEP_hourly.csv"
DATE_COL     = "Datetime"
MW_COL       = "AEP_MW"
FREQ         = "D"
SEASON_LEN   = 7
HORIZON      = 14
TEST_DAYS    = 14
VAL_DAYS     = 28
FLAML_BUDGET = 90
RANDOM_STATE = 42
print("Config OK.")

## Kaggle Credential Check

In [ ]:
import os, pathlib as _pl
_ok = False
if os.environ.get("KAGGLE_USERNAME") or os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN"):
    print("[OK] Kaggle env vars found."); _ok = True
_j = _pl.Path.home() / ".kaggle" / "kaggle.json"
if _j.exists(): print(f"[OK] kaggle.json found"); _ok = True
if not _ok:
    raise EnvironmentError("Set Kaggle credentials. See Section 11 for details.")

## Dataset Download & Loading

In [ ]:
import kagglehub
data_path = pathlib.Path(kagglehub.dataset_download(KAGGLE_SLUG))
all_files = sorted(data_path.rglob("*.csv"))
print("Available files:")
for f in all_files: print(f"  {f.name}  ({f.stat().st_size//1024:,} KB)")

# Load AEP (largest, most complete)
target_path = next((f for f in all_files if TARGET_FILE in f.name), all_files[0])
print(f"\nLoading: {target_path.name}")
df = pd.read_csv(target_path)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

## Data Validation & Resampling

In [ ]:
# Detect actual MW column (may not be exactly 'AEP_MW' depending on the file chosen)
mw_col = next((c for c in df.columns if "MW" in c.upper() or "mw" in c.lower()), None)
if mw_col is None: mw_col = df.select_dtypes(include=[np.number]).columns[0]
print(f"MW column: {mw_col}")
print(f"Date column: {DATE_COL}")

df[DATE_COL] = pd.to_datetime(df[DATE_COL], infer_datetime_format=True, errors="coerce")
df = df.dropna(subset=[DATE_COL]).sort_values(DATE_COL).reset_index(drop=True)
print(f"Date range: {df[DATE_COL].min().date()} → {df[DATE_COL].max().date()}")
print(f"Rows: {len(df):,}")
print(f"\n{mw_col} stats:")
print(df[mw_col].describe().round(1))

In [ ]:
# Resample to daily mean
daily_raw = df.set_index(DATE_COL)[mw_col].resample("D").mean().reset_index()
daily_raw.columns = ["ds","y"]
daily_raw = daily_raw.dropna().sort_values("ds").reset_index(drop=True)
# Fill any calendar gaps via linear interpolation
full_idx = pd.date_range(daily_raw["ds"].min(), daily_raw["ds"].max(), freq="D")
daily = daily_raw.set_index("ds").reindex(full_idx).reset_index()
daily.columns = ["ds","y"]
daily["y"] = daily["y"].interpolate("linear")
print(f"Daily series: {len(daily)} days  ({daily['ds'].min().date()} – {daily['ds'].max().date()})")
print(daily["y"].describe().round(1))

## EDA

In [ ]:
fig = px.line(daily, x="ds", y="y",
              title="Daily Mean Electricity Demand — AEP Region (MW)",
              labels={"ds":"Date","y":"MW"})
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
daily["year"]  = daily["ds"].dt.year
daily["month"] = daily["ds"].dt.month
daily["dow"]   = daily["ds"].dt.day_name()
dow_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

fig = px.box(daily, x="dow", y="y", category_orders={"dow":dow_order},
             title="Demand Distribution by Day of Week",
             labels={"dow":"Day","y":"MW"})
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
monthly_avg = daily.groupby("month")["y"].mean()
fig = px.bar(x=monthly_avg.index, y=monthly_avg.values,
             title="Average Demand by Month (Annual Seasonality)",
             labels={"x":"Month","y":"Avg MW"})
fig.update_layout(template="plotly_white")
fig.show()
print("Expected: dual-peak (January + July) from heating + cooling demand")

In [ ]:
if len(daily) > 3*SEASON_LEN:
    sample = daily.tail(365*3).set_index("ds")["y"].asfreq("D").interpolate()
    decomp = seasonal_decompose(sample, model="additive", period=SEASON_LEN)
    fig, axes = plt.subplots(4,1,figsize=(14,10),sharex=True)
    decomp.observed.plot(ax=axes[0], title="Observed — Last 3 Years")
    decomp.trend.plot(ax=axes[1], title="Trend")
    decomp.seasonal.plot(ax=axes[2], title="Seasonal (weekly)")
    decomp.resid.plot(ax=axes[3], title="Residual")
    plt.tight_layout(); plt.show()

In [ ]:
adf = adfuller(daily["y"].dropna())
print(f"ADF p={adf[1]:.4f} → {'Stationary' if adf[1]<0.05 else 'Non-stationary'}")
fig,axes = plt.subplots(1,2,figsize=(14,4))
plot_acf(daily["y"],  lags=90, ax=axes[0], title="ACF — Daily MW (lags 90)")
plot_pacf(daily["y"], lags=40, ax=axes[1], title="PACF — Daily MW (lags 40)")
plt.tight_layout(); plt.show()

## Target Analysis

In [ ]:
y = daily["y"]
print(f"Mean={y.mean():,.1f}  Std={y.std():,.1f}  CV={y.std()/y.mean():.3f}")
peak_day = daily.loc[y.idxmax()]
print(f"Peak demand: {y.max():,.0f} MW on {peak_day['ds'].date()}")
low_day = daily.loc[y.idxmin()]
print(f"Min demand : {y.min():,.0f} MW on {low_day['ds'].date()}")

## Train / Validation / Test Split

In [ ]:
n = len(daily); test_start = n-TEST_DAYS; val_start = test_start-VAL_DAYS
ts_train = daily.iloc[:val_start].copy(); ts_val = daily.iloc[val_start:test_start].copy()
ts_test  = daily.iloc[test_start:].copy(); ts_trainval = daily.iloc[:test_start].copy()
print(f"Train={len(ts_train)} | Val={len(ts_val)} | Test={len(ts_test)}")
assert ts_train["ds"].max() < ts_val["ds"].min()
assert ts_val["ds"].max() < ts_test["ds"].min()
print("No overlap confirmed.")

## Feature Engineering

In [ ]:
def build_feats(df_in):
    df = df_in.copy().sort_values("ds").reset_index(drop=True); y = df["y"]
    for lag in [1,7,14,28,365]:  df[f"lag_{lag}"] = y.shift(lag)
    for w in [7,14,28]:
        df[f"rmean_{w}"] = y.shift(1).rolling(w).mean()
        df[f"rstd_{w}"]  = y.shift(1).rolling(w).std()
    df["dow"]        = df["ds"].dt.dayofweek
    df["is_weekend"] = (df["dow"] >= 5).astype(int)
    df["month"] = df["ds"].dt.month; df["dayofyear"] = df["ds"].dt.dayofyear
    # Cooling degree days proxy: summer indicator
    df["is_summer"] = df["month"].isin([6,7,8]).astype(int)
    df["is_winter"] = df["month"].isin([12,1,2]).astype(int)
    return df
full_feat = build_feats(daily)
feat_train    = full_feat.iloc[:val_start].dropna().copy()
feat_val      = full_feat.iloc[val_start:test_start].dropna().copy()
feat_test     = full_feat.iloc[test_start:].dropna().copy()
feat_trainval = full_feat.iloc[:test_start].dropna().copy()
FEAT_COLS = [c for c in feat_train.columns if c not in ["ds","y"]]
print(f"Features ({len(FEAT_COLS)}): {FEAT_COLS}")

## Baselines

In [ ]:
results = []; y_test = ts_test["y"].values; last_tv = ts_trainval["y"].values
results.append(metrics(y_test, np.full(TEST_DAYS, last_tv[-1]), "Naive"))
sn = np.tile(last_tv[-SEASON_LEN:], TEST_DAYS//SEASON_LEN+1)[:TEST_DAYS]
results.append(metrics(y_test, sn, "Seasonal Naive (7-day)"))
results.append(metrics(y_test, np.full(TEST_DAYS, last_tv[-7:].mean()), "MA-7"))

## LazyPredict

In [ ]:
try:
    lz = LazyRegressor(verbose=0, ignore_warnings=True)
    lm,_ = lz.fit(feat_train[FEAT_COLS], feat_val[FEAT_COLS], feat_train["y"], feat_val["y"])
    print(lm.sort_values("RMSE").head(10).to_string())
except Exception as e: print(f"LazyPredict: {e}")

## FLAML AutoML

In [ ]:
flaml = AutoML()
flaml.fit(feat_trainval[FEAT_COLS], feat_trainval["y"], task="regression", metric="rmse",
          time_budget=FLAML_BUDGET, verbose=0, seed=RANDOM_STATE)
flaml_pred = flaml.predict(feat_test[FEAT_COLS]) if len(feat_test)>0 else None
print(f"Best: {flaml.best_estimator}")
if flaml_pred is not None:
    results.append(metrics(feat_test["y"].values, flaml_pred, f"FLAML ({flaml.best_estimator})"))

## Darts — ExponentialSmoothing & AutoARIMA

Darts provides a clean API for both classical and neural models.

**Model choices:**
- `ExponentialSmoothing(trend="additive", seasonal="additive", seasonal_periods=7)` — 
  captures the weekly seasonality and mild trend via state-space ETS
- `AutoARIMA` — selects ARIMA order automatically; good for medium-term forecasts
- `NaiveSeasonal(K=7)` — weekly seasonal naive as Darts baseline

In [ ]:
train_darts = TimeSeries.from_dataframe(ts_trainval, time_col="ds", value_cols="y", freq=FREQ)
test_darts  = TimeSeries.from_dataframe(ts_test,     time_col="ds", value_cols="y", freq=FREQ)
darts_preds = {}

# ETS (weekly seasonality)
try:
    ets_d = ExponentialSmoothing(trend="additive", seasonal="additive", seasonal_periods=SEASON_LEN)
    ets_d.fit(train_darts)
    ets_fc = ets_d.predict(HORIZON)
    darts_preds["Darts-ETS"] = ets_fc.values().flatten()[:TEST_DAYS]
    results.append(metrics(y_test, darts_preds["Darts-ETS"], "Darts-ETS"))
except Exception as e: print(f"Darts ETS: {e}")

# AutoARIMA
try:
    arima_d = DartsAutoARIMA()
    arima_d.fit(train_darts)
    arima_fc = arima_d.predict(HORIZON)
    darts_preds["Darts-AutoARIMA"] = arima_fc.values().flatten()[:TEST_DAYS]
    results.append(metrics(y_test, darts_preds["Darts-AutoARIMA"], "Darts-AutoARIMA"))
except Exception as e: print(f"Darts AutoARIMA: {e}")

# Naive Seasonal baseline
try:
    sn_d = NaiveSeasonal(K=SEASON_LEN)
    sn_d.fit(train_darts)
    sn_fc = sn_d.predict(HORIZON)
    darts_preds["Darts-SeasonalNaive"] = sn_fc.values().flatten()[:TEST_DAYS]
    results.append(metrics(y_test, darts_preds["Darts-SeasonalNaive"], "Darts-SeasonalNaive"))
except Exception as e: print(f"Darts SeasonalNaive: {e}")

## MLForecast (LightGBM)

In [ ]:
mlf_df = ts_trainval[["ds","y"]].assign(unique_id="aep")
mlf = MLForecast(
    models={"lgbm": LGBMRegressor(n_estimators=300, learning_rate=0.05, verbosity=-1,
                                    random_state=RANDOM_STATE)},
    freq=FREQ, lags=[1,7,14,28],
    lag_transforms={1:[(rolling_mean,7),(rolling_mean,28)]},
    date_features=["dayofweek","month","dayofyear"], num_threads=2)
mlf.fit(mlf_df); mlf_fc = mlf.predict(h=HORIZON)
results.append(metrics(y_test, mlf_fc["lgbm"].values[:TEST_DAYS], "MLF-LightGBM"))

## Top 3 Model Selection

In [ ]:
res_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print(res_df.to_string()); top3 = res_df.head(3)
print("\n>>> TOP 3 <<<"); print(top3.to_string(index=False))
fig = px.bar(res_df, x="Model", y="RMSE", color="RMSE", color_continuous_scale="RdYlGn_r",
             title="Power Demand — Model Comparison (RMSE)")
fig.update_layout(template="plotly_white", xaxis_tickangle=-35)
fig.show()

## Final Forecast Plot

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_trainval["ds"].tail(90), y=ts_trainval["y"].tail(90),
    name="Train (90 days)", line=dict(color="royalblue", dash="dot", width=1)))
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"],
    name="Actual", line=dict(color="black", width=2)))
all_preds = {}
if flaml_pred is not None: all_preds[f"FLAML ({flaml.best_estimator})"] = flaml_pred
all_preds.update(darts_preds)
if "lgbm" in mlf_fc.columns: all_preds["MLF-LightGBM"] = mlf_fc["lgbm"].values[:TEST_DAYS]
all_preds["Seasonal Naive (7-day)"] = sn
colors = ["#e15759","#f28e2b","#59a14f"]
for i,(_,row) in enumerate(top3.iterrows()):
    m = row["Model"]
    if m in all_preds:
        fig.add_trace(go.Scatter(x=ts_test["ds"][:TEST_DAYS], y=all_preds[m],
            name=f"#{i+1} {m}", line=dict(color=colors[i], width=2)))
fig.update_layout(title="Top 3 Power Demand Forecasts vs Actual",
                  template="plotly_white", xaxis_title="Date", yaxis_title="MW")
fig.show()

## Error Analysis

In [ ]:
best_m = top3.iloc[0]["Model"]
if best_m in all_preds:
    bp = np.asarray(all_preds[best_m]).ravel(); ba = y_test[:len(bp)]; err = ba-bp
    print(f"Bias={err.mean():+.1f}  Std={err.std():.1f}")
    fig,ax = plt.subplots(1,2,figsize=(14,4))
    ax[0].hist(err, bins=14, color="steelblue", edgecolor="white")
    ax[0].axvline(0,color="red",linestyle="--"); ax[0].set_title("Residual Distribution")
    ax[1].scatter(ba, bp, s=50, alpha=0.7, color="steelblue")
    mx = max(ba.max(),bp.max())*1.1; ax[1].plot([0,mx],[0,mx],"r--")
    ax[1].set_xlabel("Actual"); ax[1].set_ylabel("Predicted"); ax[1].set_title("Actual vs Predicted")
    plt.tight_layout(); plt.show()

## Interpretation & Insights
1. **Weekly pattern dominates short-term** (7 days): industrial & commercial load halves on weekends
2. **Annual pattern dominates long-term** (365 days): July peak (AC) + January peak (heating)
3. **Darts ETS** with additive seasonality captures the weekly cycle directly in the state-space
4. **LightGBM with lag features** often matches or beats ETS when lag-365 is available
5. **Forecast error > 5% of peak** is operationally problematic — grid operators need narrow confidence


## Limitations
1. Daily resampling loses the extreme within-day peak (4–8pm) that stresses grid lines
2. No temperature features — electricity demand is strongly correlated with heat/cold
3. No holiday calendar — US holidays are significant demand-reduction events
4. Single-region model — AEP covers many states with very different weather patterns

## How to Improve
1. Add NOAA daily temperature (HDD/CDD) as exogenous covariate — 5-10% error reduction expected
2. Use hourly forecasting with Darts N-HiTS or PatchTST for intraday grid dispatch
3. Add public holiday indicator (US federal holidays)
4. Hierarchical forecasting: forecast at state level and aggregate to AEP total

## Production Considerations
1. **Hourly resolution**: Production systems forecast hourly (not daily) for intraday dispatch scheduling
2. **Weather covariates**: Integrate NOAA/NWS temperature forecasts as exogenous inputs — electricity demand is ~70% driven by weather
3. **Probabilistic forecasts**: Use quantile regression or conformal prediction to produce 10th/50th/90th percentile bands
4. **Rolling retrain**: Retrain daily with latest data; power demand patterns shift with economic activity and efficiency trends
5. **Regulatory reporting**: Grid operators (ISO/RTO) require next-day forecasts by 10am — pipeline latency matters
6. **Fallback strategy**: If the primary model fails, use the same-day-last-week pattern as an emergency forecast

## Common Mistakes
1. Using the raw hourly series without flagging DST transition hours (23/25-hour days)
2. Conflating "energy" (MWh) and "power" (MW) — this dataset is power (instantaneous)
3. Using MAPE without care — MW values are large; MAPE is naturally small and flattering
4. Treating annual seasonality the same as weekly (different period, different model)

## Exercises
1. Rebuild with hourly data and compare to daily: does finer resolution improve 14-day forecasts?
2. Add NOAA GHCND temperature data as a column and re-run FLAML; measure improvement
3. Try Darts `KalmanForecaster` for its noise-robust state-space formulation
4. Plot the 14-day rolling MAE over the last year to check forecast stability

## Summary
- Loaded 13-years of hourly AEP electricity consumption; resampled to daily mean
- EDA: clear dual-peak annual pattern (summer AC, winter heating) + strong weekly pattern
- Compared Baselines → LazyPredict → FLAML → Darts (ETS + AutoARIMA) → MLForecast
- Key lesson: Daily electricity demand is highly forecastable (CV~15%) with seasonal models

---
*Notebook #16 of 50 — Time Series Forecasting Portfolio*